# Retail Sales Data Analytics: Data Cleaning

## Project Overview

This project focuses on cleaning and validating a retail sales dataset to prepare it for further analysis and modeling. The dataset contains transaction-level retail information, including invoice numbers, stock codes, product descriptions, quantities, invoice dates, unit prices, customer IDs, and countries.

The data-cleaning process focuses on improving data quality while preserving useful information. The main steps include inspecting the dataset, standardizing column names, validating data types, handling missing values, removing duplicate records, validating geographical information, and saving the final cleaned dataset.

After completing these steps, the cleaned dataset is saved in the `data/cleaned` directory and can be used as the foundation for exploratory data analysis, visualization, and future modeling.

## Document Purpose

The purpose of this document is to provide a clear and reproducible record of the data-cleaning process. It explains the decisions made during cleaning, the reasoning behind those decisions, and the validation performed to ensure that the resulting dataset is reliable and suitable for the next stage of the project.

This document covers the following areas:

- **Dataset Preview:** Inspect the dataset structure, dimensions, and initial data types.
- **Column Standardization:** Convert column names to a consistent naming convention.
- **Data Type Validation:** Investigate identifier columns and avoid inappropriate type conversions that could result in errors or information loss.
- **Missing Value Handling:** Identify, investigate, and appropriately handle missing values.
- **Duplicate Detection:** Identify and remove duplicate transaction records.
- **Geographical Data Validation:** Inspect country labels for inconsistencies or unexpected values.
- **Final Validation and Export:** Review the cleaned dataset and save it to the `data/cleaned` directory.

The goal is not simply to remove unwanted data, but to make **evidence-based cleaning decisions** that preserve the integrity and usefulness of the original dataset.

## 1. Dataset Preview

Before we start cleaning our dataset, we first need to import the required libraries and load the dataset. This prepares our environment for the data cleaning process.

In [140]:
# modules we'll use
import os
import pandas as pd
import numpy as np

# set project root
project_root = os.getcwd()
if os.path.basename(project_root) == "noteboook":
    project_root = os.path.dirname(project_root)

file_path = os.path.join(project_root, "data", "raw", "retail_sales.xlsx")

try:
    df = pd.read_excel(file_path)
except ImportError as error:
    raise ImportError("Install openpyxl first: pip install openpyxl") from error

# set seed for reproducibility
np.random.seed(0)

Great! Now let’s take a look at our dataset. This helps us confirm that the data was loaded correctly and gives us a better understanding of its structure and contents.

In [141]:
df.shape

(541909, 8)

The dataset contains over `540k rows` and `8 columns`. Next, let's inspect the column names and their data types.

In [142]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


It looks like we need to standardize our column names. We should also investigate the `InvoiceNo` and `StockCode` columns, as they are currently stored as an `object` data type. Let's take a closer look at the dataset to better understand these issues.

In [143]:
# preview first 10 rows of the dataset
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


Now we understand why the `StockCode` column is stored as an `object` data type. Although some values appear numeric, the column also contains alphabetic characters, meaning it represents identifiers rather than numerical values. Therefore, converting it to an `int` data type would be inappropriate.

## 2. Column Standardization

Before starting the data cleaning process, we need to standardize our column names to maintain consistency throughout the analysis. Currently, some columns such as `InvoiceNo`, `StockCode`, and `Description` use uppercase letters. We will convert all column names to lowercase and replace spaces with underscores (`_`) to follow a consistent naming convention.

In [144]:
# check every columns name
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [145]:
# turn column name to lowercase
df.columns = df.columns.str.lower()

# separate column names with underscore ('_')
df = df.rename(columns={
    'invoiceno'     : 'invoice_no',
    'stockcode'     : 'stock_code',
    'invoicedate'   : 'invoice_date',
    'unitprice'     : 'unit_price',
    'customerid'    : 'customer_id'
})

# verify the changes
df.columns

Index(['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date',
       'unit_price', 'customer_id', 'country'],
      dtype='str')

## 3. Data Type Validation

As mentioned above, the `invoice_no` and `stock_code` columns are both stored as `object` data types. However, `stock_code` contains both numeric and alphabetic characters, so converting it to an integer would result in errors and potentially cause us to lose information.

Therefore, we will keep `stock_code` as it is and convert only `invoice_no` to an integer, since the `invoice_no` column contains only numeric values. However, before making this conversion, we need to verify that all values in `invoice_no` are actually numeric.

In [146]:
# verify invoice_no column
# returns True if every single value is numeric, otherwise False
is_all_numeric = pd.to_numeric(df["invoice_no"], errors="coerce").notna().all()
print(is_all_numeric)

False


In [147]:
# take a look at the column that are not numeric
non_numeric_series = df[pd.to_numeric(df["invoice_no"], errors="coerce").isna()]["invoice_no"]
print(non_numeric_series)

141       C536379
154       C536383
235       C536391
236       C536391
237       C536391
           ...   
540449    C581490
541541    C581499
541715    C581568
541716    C581569
541717    C581569
Name: invoice_no, Length: 9291, dtype: object


After verifying our `invoice_no` column, we can see that it also contains both numeric and alphabetic characters, similar to `stock_code`. Therefore, converting it to an integer would result in errors and could cause us to lose useful information.

## 4. Missing Value Handling
A dataset with missing values can lead to inaccurate results during **Analysis** and **Modeling**. To ensure that our dataset is reliable and ready to use, we need to check for and verify whether it contains any **Null** values.

In [148]:
# check number of missing value
missing_value_count = df.isnull().sum()
missing_value_count

invoice_no           0
stock_code           0
description       1454
quantity             0
invoice_date         0
unit_price           0
customer_id     135080
country              0
dtype: int64

We found missing values in the `customer_id` and `description` columns. Let's determine the total number of missing values in the dataset.

In [149]:
# calculate number of missing values 
total_cells = np.prod(df.shape)
total_missing = missing_value_count.sum()
percent_missing = (total_missing/total_cells) * 100

print("Total values:", df.size)
print("Total missing values:", total_missing)
print("Percentage missing:", percent_missing)

Total values: 4335272
Total missing values: 136534
Percentage missing: 3.149375633178264


The dataset contains over 130K missing values, which account for only 3.14% of the total values. Although this is a relatively small percentage, we should not immediately decide whether to drop or fill the missing values.

First, we need to inspect the missing values to understand their distribution and determine whether the affected columns contain useful information. We should also check whether the missing values occur in the same rows across both columns. This will help us choose the most appropriate strategy for handling them.

In [150]:
# check what happen when customer_id missing
missing_customer_id = df[df['customer_id'].isnull()]

# view the dataset
missing_customer_id.head(10)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1448,536544,21791,VINTAGE HEADS AND TAILS CARD GAME,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1449,536544,21801,CHRISTMAS TREE DECORATION WITH BELL,10,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1450,536544,21802,CHRISTMAS TREE HEART DECORATION,9,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1451,536544,21803,CHRISTMAS TREE STAR DECORATION,11,2010-12-01 14:32:00,0.43,NaN,United Kingdom


In [151]:
# check what happen when description missing
missing_description = df[df['description'].isnull()]

# view the dataset
missing_description.head(10)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


In [152]:
# check if both missing the same row
missing_both_value = df[df[['customer_id', 'description']].isnull().all(axis=1)]

# view the dataset
missing_both_value.head(10)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


We can see that both columns have missing values in the same rows. Let's determine how many rows are affected by missing values in both columns.

In [153]:
# filter for rows where both columns are missing
missing_both = df[df['customer_id'].isnull() & df['description'].isnull()]

# total count of affected rows
affected_rows_count = missing_both.shape[0]

print(f"Number of rows missing both customer_id and description: {affected_rows_count}")


Number of rows missing both customer_id and description: 1454


Now we have a better understanding of the missing values. Both `customer_id` and `description` have the same number of missing values, and these missing values occur in exactly the same rows. In other words, every row where `description` is missing also has a missing `customer_id`.

However, when we look at the other columns, such as `stock_code`, `quantity`, `invoice_date`, and `unit_price`, they still contain useful information. We also noticed some suspicious patterns. For example, some rows have a `unit_price` of `0.0`, while the `invoice_date` and `country` are the same, even though the `invoice_no` values are different.

In addition, some of these rows have negative values in the `quantity` column. These patterns suggest that these records may represent unusual or system-generated transactions rather than normal customer purchases.

Based on these observations, we have enough evidence to consider removing these rows because they contain missing `customer_id` and `description` values along with other unusual transaction patterns.

In [154]:
# drops rows where both customer_id and description are missing at the same time
df.dropna(subset=['customer_id', 'description'], how='all')

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


Now, we have removed the rows where both `customer_id` and `description` were missing at the same time. The only remaining missing values are in the `customer_id` column.

Since each customer has a unique `CustomerID`, we cannot accurately recover these missing IDs. Filling them with estimated values could introduce incorrect customer information and negatively affect our analysis.

Therefore, we will keep these missing values as Pandas `NaN` instead of replacing them with `Unknown Customer`. This is because `customer_id` is currently stored as a float data type, and using a text value would change the column type and make it inconsistent. Keeping `NaN` allows us to represent unknown customer information without modifying the original data structure.

## 5. Duplicate Detection

Before the dataset is ready for analysis and modeling, we need to check whether it contains any duplicate values that could lead to inaccurate results. To ensure the dataset is reliable and prepared for the next steps, we need to identify and verify any duplicate records.

In [155]:
# count the total number of duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows: {duplicate_count}")

Total duplicate rows: 5268


There are over 5,000 duplicate rows in the entire dataset. Although this is a small number compared to the total number of rows, duplicate records can still lead to inaccurate results during analysis and modeling. Our next step is to inspect these duplicate rows to understand what they look like and determine whether they should be removed.

In [ ]:
# preview the first 10 duplicate rows
duplicates = df[df.duplicated(keep=False)]
duplicates.head(10)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom


Now we can clearly see the duplicate records. For example, rows 485 and 539, as well as rows 494 and 539, contain the same values for `stock_code`, `description`, ` uantity`, `invoice_date`, `unit_price`, `customer_id`, and `country`.

This evidence confirms that these records are duplicates. Our next step is to remove the duplicate rows and then verify that no duplicate records remain in the dataset.

In [157]:
# drop duplicates rows
df.drop_duplicates(inplace=True)

In [159]:
# verify after drop duplicate
verify_dup = df.duplicated().sum()
print(f"Verify duplicate rows: {verify_dup}")

Verify duplicate rows: 0


## 6. Geographical Data Validation

Before moving forward with our analysis, we need to validate the geographical information in the dataset. The `country` column contains the country associated with each transaction, so it is important to make sure that the values are consistent and correctly represented.

We will first inspect the unique country names and check for any inconsistent spelling, unexpected values, or formatting issues. This will help ensure that the geographical data is clean and reliable for further analysis, especially when comparing sales across different countries.

In [163]:
df['country'].unique()

<ArrowStringArray>
[      'United Kingdom',               'France',            'Australia',
          'Netherlands',              'Germany',               'Norway',
                 'EIRE',          'Switzerland',                'Spain',
               'Poland',             'Portugal',                'Italy',
              'Belgium',            'Lithuania',                'Japan',
              'Iceland',      'Channel Islands',              'Denmark',
               'Cyprus',               'Sweden',              'Austria',
               'Israel',              'Finland',              'Bahrain',
               'Greece',            'Hong Kong',            'Singapore',
              'Lebanon', 'United Arab Emirates',         'Saudi Arabia',
       'Czech Republic',               'Canada',          'Unspecified',
               'Brazil',                  'USA',   'European Community',
                'Malta',                  'RSA']
Length: 38, dtype: str

After inspecting the `country` column, everything appears to be consistent and correctly formatted. The country names are valid, and we did not find any obvious inconsistencies or incorrect values. Therefore, no further cleaning is required for the geographical data.

### Geographical Data Insights

- **Geographical Scope:** The dataset contains 38 unique country labels.
- **Standard Acronyms:** The values `EIRE` (Ireland) and `RSA` (Republic of South Africa) represent specific naming conventions rather than data corruption.
- **Aggregated Labels:** `European Community` and `Unspecified` are valid system-level labels that may represent anonymous, regional, or bulk distributions.

Based on these findings, we can keep the `Country` column as it is and proceed to the next stage of data cleaning.

## 7. Final Validation and Export

We have completed the data cleaning process, and our dataset is now ready for analysis and modeling. As a final step, we will save the cleaned dataset to the `data/cleaned` directory.

Before saving it, let's take one final look at the dataset to confirm that everything is correct and that all cleaning steps have been applied successfully.

In [164]:
# final preview
df.head(20)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


We have confirmed that the dataset is clean and ready to use. Let's save the cleaned dataset.

In [166]:
# module we'll use
from pathlib import Path

# read output directory if it doesn't exist
output_dir = Path("../data/cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

# refine the clean file path
CLEAN_DATA_PATH = output_dir / "cleaned_retail_sales.xlsx"

# save the dataset
df.to_excel(CLEAN_DATA_PATH, index=False)
print(f"Success! Cleaned dataset saved to: {CLEAN_DATA_PATH}")

Success! Cleaned dataset saved to: ../data/cleaned/cleaned_retail_sales.xlsx


## Summary

The data-cleaning process has been completed successfully. We inspected the dataset, standardized the column names, validated data types, handled missing values, removed duplicate records, and validated the geographical data.

During the cleaning process, we found that some identifier columns contained both numeric and alphabetic values, so they were kept as `object` data types to avoid losing information. Missing `customer_id` and `description` values were investigated, and rows where both were missing were removed. The remaining missing `customer_id` values were kept as `NaN` because the original customer IDs could not be reliably recovered. We also identified and removed duplicate transaction records.

Finally, the cleaned dataset was validated and saved to the `data/cleaned` directory. The dataset is now ready for the next stage of the project, including exploratory data analysis, visualization, and modeling.